# 01 — Data Exploration

Before building any model we need to understand the data:
- How many rows / columns in each table?
- What does each column actually mean?
- Where are the missing values?
- What patterns exist that a model could learn from?

This process is called **EDA — Exploratory Data Analysis**.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import sys
from pathlib import Path

# Robust project root finder — works no matter where Jupyter was launched from.
# It walks UP the directory tree from wherever the notebook is running until
# it finds a folder that contains both "src/" and "requirements.txt".
# That folder is guaranteed to be the project root.
def find_project_root(start: Path, depth: int = 5) -> Path:
    path = start.resolve()
    for _ in range(depth):
        if (path / "src").exists() and (path / "requirements.txt").exists():
            return path
        path = path.parent
    raise RuntimeError(
        f"Could not find project root from {start}. "
        "Make sure src/ and requirements.txt exist in the project root."
    )

PROJECT_ROOT = find_project_root(Path.cwd())
FIGURES_DIR  = PROJECT_ROOT / "outputs" / "figures"
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

sys.path.insert(0, str(PROJECT_ROOT))
from src.data_loader import load_raw_tables, build_master_df, add_parsed_lap_times

plt.style.use("seaborn-v0_8-darkgrid")
pd.set_option("display.max_columns", 50)

print("Project root  :", PROJECT_ROOT)
print("Figures saved :", FIGURES_DIR)

## Load the raw tables

In [ ]:
tables = load_raw_tables()

for name, df in tables.items():
    print(f"{name:30s}  shape: {df.shape}")

## Inspect results.csv — our most important table

Every row = one driver's result in one race.  
`positionOrder` is the final classified position — this is what we predict.

In [ ]:
results = tables["results"]
print(results.dtypes)
print()
results.head(10)

In [ ]:
missing = results.isnull().sum()
print("Missing values in results.csv:")
print(missing[missing > 0])

## Key question 1: Does grid position predict podium?

Starting from pole (P1) puts you ahead of everyone.  
Let's measure exactly how strong this signal is in the data.

In [ ]:
df = build_master_df(tables)
df = add_parsed_lap_times(df)

df["is_podium"]     = (df["positionOrder"] <= 3).astype(int)
df["grid"]          = pd.to_numeric(df["grid"], errors="coerce")
df["grid_position"] = df["grid"].replace(0, 20)

podium_by_grid = (
    df[df["grid_position"].between(1, 20)]
    .groupby("grid_position")["is_podium"]
    .mean()
    .reset_index()
)

fig, ax = plt.subplots(figsize=(10, 5))
ax.bar(podium_by_grid["grid_position"], podium_by_grid["is_podium"], color="steelblue")
ax.set_xlabel("Starting Grid Position")
ax.set_ylabel("Podium Rate")
ax.set_title("Podium Rate by Starting Grid Position (1950-2024)")
ax.set_xticks(range(1, 21))
plt.tight_layout()
plt.savefig(FIGURES_DIR / "podium_rate_by_grid.png", dpi=150)
plt.show()

## Key question 2: Class balance

Only 3 of ~20 drivers get a podium per race — roughly **13%** of all rows.  
This is called **class imbalance**. A model that always predicts "no podium"  
would still be 87% accurate — but completely useless.  
We handle this with `class_weight="balanced"` in the model.

In [ ]:
balance = df["is_podium"].value_counts(normalize=True)
print("Class distribution:")
print(balance.rename({0: "No Podium", 1: "Podium"}))

fig, ax = plt.subplots(figsize=(5, 4))
ax.bar(["No Podium", "Podium"], balance.values, color=["#e74c3c", "#2ecc71"])
ax.set_ylabel("Proportion of rows")
ax.set_title("Target Variable Class Balance")
plt.tight_layout()
plt.savefig(FIGURES_DIR / "class_balance.png", dpi=150)
plt.show()

## Key question 3: How much qualifying data is available?

Q1/Q2/Q3 format only started in **2006**.  
Before that, everyone did a single flying lap, so there are no Q3 times.

In [ ]:
for col in ["q1_seconds", "q2_seconds", "q3_seconds"]:
    if col in df.columns:
        missing_pct = df[col].isna().mean()
        print(f"{col}: {missing_pct:.1%} missing")

## Key question 4: Which constructors dominate?

In F1, the car matters as much as the driver.  
Let's see which teams have the most wins across all of history.

In [ ]:
constructor_wins = (
    df[df["positionOrder"] == 1]
    .groupby("constructorRef")
    .size()
    .sort_values(ascending=False)
    .head(15)
)

fig, ax = plt.subplots(figsize=(10, 5))
constructor_wins.plot(kind="bar", ax=ax, color="tomato")
ax.set_xlabel("Constructor")
ax.set_ylabel("Race Wins")
ax.set_title("Top 15 Constructors by Race Wins (1950-2024)")
ax.tick_params(axis="x", rotation=45)
plt.tight_layout()
plt.savefig(FIGURES_DIR / "constructor_wins.png", dpi=150)
plt.show()

## Key question 5: Races per season over time

Seasons grew from 7 races in 1950 to 24 in recent years.  
This matters for rolling-average features — more races = smoother form estimates.

In [ ]:
races_per_year = df.groupby("year")["raceId"].nunique()

fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(races_per_year.index, races_per_year.values,
        marker="o", markersize=3, color="steelblue")
ax.set_xlabel("Year")
ax.set_ylabel("Number of Races")
ax.set_title("F1 Races Per Season (1950-2024)")
plt.tight_layout()
plt.savefig(FIGURES_DIR / "races_per_year.png", dpi=150)
plt.show()

print("EDA complete. All figures saved to:", FIGURES_DIR)